In [30]:
import pandas as pd
import numpy as np
import json
import os
from pathlib import Path
import sklearn 
from sklearn.model_selection import train_test_split


In [31]:
def load_session(filepath):
    df = pd.read_csv(filepath)
    
    # drop scroll events
    df = df[df['button'] != 'Scroll']
    
    # drop 65535 coordinate artifacts
    df = df[~((df['x'] == 65535) | (df['y'] == 65535))]
    
    # sort by record timestamp
    df = df.sort_values('record timestamp').reset_index(drop=True)
    
    return df

In [32]:
cleaned_sessions = load_session('DataSamples/session_2144641057.txt')
print(f"Total rows: {cleaned_sessions.shape[0]}")
print(cleaned_sessions.head())
print("\n")

# event distribution
print(cleaned_sessions['state'].value_counts())
print(cleaned_sessions['button'].value_counts())
print("\n")

# left clicks
left_clicks = cleaned_sessions[
    (cleaned_sessions['button'] == 'Left') & 
    (cleaned_sessions['state'] == 'Pressed')
]
print(f"Total left clicks: {len(left_clicks)}")
print("\n")

# timestamp gaps
moves = cleaned_sessions[cleaned_sessions['state'] == 'Move'].copy()
moves['dt'] = moves['record timestamp'].diff()
print(f"Median gap: {moves['dt'].median()*1000:.1f}ms")
print(f"Gaps over 2s: {len(moves[moves['dt'] > 2.0])}")
print(f"Largest gap: {moves['dt'].max():.1f}s")
print("\n")

# coordinate range
print(f"X range: {cleaned_sessions['x'].min()} to {cleaned_sessions['x'].max()}")
print(f"Y range: {cleaned_sessions['y'].min()} to {cleaned_sessions['y'].max()}")


Total rows: 29675
   record timestamp  client timestamp    button state     x    y
0             0.000             0.000  NoButton  Move  1043  410
1             0.232             0.094  NoButton  Move  1024  410
2             0.232             0.094  NoButton  Move  1019  410
3             0.232             0.234  NoButton  Move   979  409
4             0.342             0.234  NoButton  Move   974  411


state
Move        26257
Drag         1258
Pressed      1080
Released     1080
Name: count, dtype: int64
button
NoButton    27515
Left         2148
Right          12
Name: count, dtype: int64


Total left clicks: 1074


Median gap: 113.0ms
Gaps over 2s: 678
Largest gap: 686.4s


X range: 0 to 1919
Y range: 0 to 1079


In [33]:
def extract_trajectories(df):
    trajectories = []
    
    press_indices = df[
        (df['state'] == 'Pressed') & 
        (df['button'] == 'Left')
    ].index.tolist()
    
    for press_idx in press_indices:
        press_row = df.loc[press_idx]
        target_x = press_row['x']
        target_y = press_row['y']
        press_time = press_row['record timestamp']
        
        points = []
        
        for i in range(press_idx - 1, max(0, press_idx - 500), -1):
            row = df.loc[i]
            
            # stop conditions
            if row['state'] in ['Pressed', 'Released']:
                break
            if row['state'] not in ['Move', 'Drag']:
                continue
            if press_time - row['record timestamp'] > 2.0:
                break
                
            points.append({
                't': row['record timestamp'],
                'x': row['x'],
                'y': row['y']
            })
        
        # reverse since collected backwards
        points = points[::-1]
        
        # filters
        if len(points) < 5:
            continue
            
        distance = np.sqrt(
            (target_x - points[0]['x'])**2 + 
            (target_y - points[0]['y'])**2
        )
        
        if distance < 50:
            continue
            
        trajectories.append({
            'target_x': int(target_x),
            'target_y': int(target_y),
            'start_x': int(points[0]['x']),
            'start_y': int(points[0]['y']),
            'distance': float(distance),
            'n_points': len(points),
            'points': points
        })
    
    return trajectories

In [34]:
trajectories = extract_trajectories(cleaned_sessions)
print(f"Trajectories extracted: {len(trajectories)}")
print(f"Sample trajectory: {trajectories[0]}")

Trajectories extracted: 622
Sample trajectory: {'target_x': 647, 'target_y': 738, 'start_x': 700, 'start_y': 614, 'distance': 134.8517704741024, 'n_points': 13, 'points': [{'t': np.float64(8.08699989319), 'x': np.int64(700), 'y': np.int64(614)}, {'t': np.float64(8.08699989319), 'x': np.int64(701), 'y': np.int64(597)}, {'t': np.float64(8.19099998474), 'x': np.int64(700), 'y': np.int64(621)}, {'t': np.float64(8.29499983788), 'x': np.int64(701), 'y': np.int64(644)}, {'t': np.float64(8.40699982643), 'x': np.int64(699), 'y': np.int64(648)}, {'t': np.float64(8.40799999237), 'x': np.int64(597), 'y': np.int64(719)}, {'t': np.float64(8.53499984741), 'x': np.int64(585), 'y': np.int64(739)}, {'t': np.float64(8.64699983597), 'x': np.int64(608), 'y': np.int64(747)}, {'t': np.float64(8.77499985695), 'x': np.int64(616), 'y': np.int64(748)}, {'t': np.float64(8.92699980736), 'x': np.int64(642), 'y': np.int64(745)}, {'t': np.float64(9.10299992561), 'x': np.int64(644), 'y': np.int64(743)}, {'t': np.float

In [35]:
import numpy as np

distances = [t['distance'] for t in trajectories]
durations = [t['points'][-1]['t'] - t['points'][0]['t'] for t in trajectories]
point_counts = [t['n_points'] for t in trajectories]

print(f"Trajectory count: {len(trajectories)}")
print(f"\nDistance stats:")
print(f"  Min: {np.min(distances):.1f}px")
print(f"  Max: {np.max(distances):.1f}px")
print(f"  Mean: {np.mean(distances):.1f}px")

print(f"\nDuration stats:")
print(f"  Min: {np.min(durations):.2f}s")
print(f"  Max: {np.max(durations):.2f}s")
print(f"  Mean: {np.mean(durations):.2f}s")

print(f"\nPoints per trajectory:")
print(f"  Min: {np.min(point_counts)}")
print(f"  Max: {np.max(point_counts)}")
print(f"  Mean: {np.mean(point_counts):.1f}")

Trajectory count: 622

Distance stats:
  Min: 50.0px
  Max: 1861.3px
  Mean: 370.1px

Duration stats:
  Min: 0.31s
  Max: 2.00s
  Mean: 1.45s

Points per trajectory:
  Min: 5
  Max: 25
  Mean: 12.0


In [36]:
def compute_features(trajectory):
    points = trajectory['points']
    target_x = trajectory['target_x']
    target_y = trajectory['target_y']
    
    featured_points = []
    
    for i, p in enumerate(points):
        if i == 0:
            dt = 0
            vx, vy = 0, 0
            ax, ay = 0, 0
            curvature = 0
            angular_v = 0
        elif i == 1:
            dt = p['t'] - points[i-1]['t']
            if dt < 0.005:
                continue
            vx = (p['x'] - points[i-1]['x']) / dt
            vy = (p['y'] - points[i-1]['y']) / dt
            ax, ay = 0, 0
            curvature = 0
            angular_v = 0
        else:
            dt = p['t'] - points[i-1]['t']
            if dt < 0.005:
                continue
            vx = (p['x'] - points[i-1]['x']) / dt
            vy = (p['y'] - points[i-1]['y']) / dt
            prev_vx = featured_points[-1]['vx']
            prev_vy = featured_points[-1]['vy']
            ax = (vx - prev_vx) / dt
            ay = (vy - prev_vy) / dt
            denom = (vx**2 + vy**2)**1.5
            curvature = abs(vx*ay - vy*ax) / denom if denom > 1e-10 else 0
            prev_angle = np.arctan2(prev_vy, prev_vx)
            curr_angle = np.arctan2(vy, vx)
            angular_v = (curr_angle - prev_angle) / dt

        featured_points.append({
            't': float(p['t']),
            'x': int(p['x']),
            'y': int(p['y']),
            'dt': dt,
            'vx': vx,
            'vy': vy,
            'speed': float(np.sqrt(vx**2 + vy**2)),
            'ax': ax,
            'ay': ay,
            'curvature': curvature,
            'angular_v': angular_v,
            'dist_to_target': float(np.sqrt(
                (p['x'] - target_x)**2 + 
                (p['y'] - target_y)**2
            ))
        })
    
    trajectory['points'] = featured_points
    return trajectory

In [37]:
trajectories = [compute_features(t) for t in trajectories]
print(f"Sample featured trajectory: {trajectories[0]}")


Sample featured trajectory: {'target_x': 647, 'target_y': 738, 'start_x': 700, 'start_y': 614, 'distance': 134.8517704741024, 'n_points': 13, 'points': [{'t': 8.08699989319, 'x': 700, 'y': 614, 'dt': 0, 'vx': 0, 'vy': 0, 'speed': 0.0, 'ax': 0, 'ay': 0, 'curvature': 0, 'angular_v': 0, 'dist_to_target': 134.8517704741024}, {'t': 8.19099998474, 'x': 700, 'y': 621, 'dt': np.float64(0.10400009154999879), 'vx': np.float64(-9.61537615108005), 'vy': np.float64(230.76902762592115), 'speed': 230.96926109320245, 'ax': np.float64(-92.45545852675897), 'ay': np.float64(2218.931004642215), 'curvature': np.float64(0.0), 'angular_v': np.float64(15.504206600801824), 'dist_to_target': 128.4445405612866}, {'t': 8.29499983788, 'x': 701, 'y': 644, 'dt': np.float64(0.10399985314000126), 'vx': np.float64(9.615398193436217), 'vy': np.float64(221.154158449033), 'speed': 221.3630901521713, 'ax': np.float64(184.91155289064125), 'ay': np.float64(-92.45079571357601), 'curvature': np.float64(0.0038519714357256123), 

In [38]:
speeds = [p['speed'] for t in trajectories for p in t['points']]
print(f"Max speed across all trajectories: {max(speeds):.1f} px/s")
print(f"Mean speed: {np.mean(speeds):.1f} px/s")
print(f"Trajectories with speed over 5000: {sum(1 for s in speeds if s > 5000)}")

Max speed across all trajectories: 173307.7 px/s
Mean speed: 388.2 px/s
Trajectories with speed over 5000: 13


In [39]:
# Filter out trajectories that have any speed over 5000
cleaned_trajectories = [
    t for t in trajectories 
    if not any(p['speed'] > 5000 for p in t['points'])
]

# Recompute speeds from cleaned trajectories
cleaned_speeds = [p['speed'] for t in cleaned_trajectories for p in t['points']]

print(f"Original trajectories: {len(trajectories)}")
print(f"Cleaned trajectories: {len(cleaned_trajectories)}")
print(f"Trajectories with speed over 5000: {len(trajectories) - len(cleaned_trajectories)}")
print(f"Max speed in cleaned data: {max(cleaned_speeds) if cleaned_speeds else 0:.1f} px/s")

Original trajectories: 622
Cleaned trajectories: 609
Trajectories with speed over 5000: 13
Max speed in cleaned data: 4918.6 px/s


In [40]:
flattened_data =[]
for traj_id, traj in enumerate(cleaned_trajectories):
    for point in traj['points']:
         row = {
            'trajectory_id': traj_id,
            'target_x': traj['target_x'],
            'target_y': traj['target_y'],
            'start_x': traj['start_x'],
            'start_y': traj['start_y'],
            'distance': traj['distance'],
            'n_points': traj['n_points'],
            't': point['t'],
            'x': point['x'],
            'y': point['y'],
            'dt': point['dt'],
            'vx': point['vx'],
            'vy': point['vy'],
            'speed': point['speed'],
            'ax': point['ax'],
            'ay': point['ay'],
            'curvature': point['curvature'],
            'angular_v': point['angular_v'],
            'dist_to_target': point['dist_to_target']
         }
         flattened_data.append(row)

df_flattened = pd.DataFrame(flattened_data)
print(df_flattened.head())

   trajectory_id  target_x  target_y  start_x  start_y   distance  n_points  \
0              0       647       738      700      614  134.85177        13   
1              0       647       738      700      614  134.85177        13   
2              0       647       738      700      614  134.85177        13   
3              0       647       738      700      614  134.85177        13   
4              0       647       738      700      614  134.85177        13   

       t    x    y     dt         vx          vy       speed          ax  \
0  8.087  700  614  0.000   0.000000    0.000000    0.000000    0.000000   
1  8.191  700  621  0.104  -9.615376  230.769028  230.969261  -92.455459   
2  8.295  701  644  0.104   9.615398  221.154158  221.363090  184.911553   
3  8.407  699  648  0.112 -17.857145   35.714289   39.929789 -245.290586   
4  8.535  585  739  0.127 -94.488297  157.480495  183.652238 -603.395587   

            ay  curvature  angular_v  dist_to_target  
0     0.00000

In [41]:
df_flattened.to_csv('../models/processed_trajectories.csv')
print(df_flattened.shape)

(6004, 19)


In [42]:
df = pd.read_csv('../models/processed_trajectories.csv')

traj_df = df.groupby('trajectory_id').agg({
    'speed': ['mean', 'std'],
    'curvature': ['mean','max'],
    'distance': 'first',
    'n_points': 'first'
}).reset_index()
traj_df.columns = ['trajectory_id', 'speed_mean', 'speed_std', 'curvature_mean', 'curvature_max', 'distance', 'n_points']
print(traj_df.head())

   trajectory_id  speed_mean   speed_std  curvature_mean  curvature_max  \
0              0  108.794655   94.897087        0.291253       2.353924   
1              1   75.025307  104.065859        0.252013       1.851948   
2              2  226.837329  259.548784        0.115089       0.638204   
3              3  556.089397  506.646184        0.042568       0.301037   
4              4  376.443255  671.542073        0.054602       0.240766   

     distance  n_points  
0  134.851770        13  
1  191.887988        13  
2  336.942132        18  
3   97.329338        17  
4  504.834626        17  


In [43]:
np.random.seed(42)
traj_df['label'] = np.random.choice([0, 1], size=len(traj_df))

X = traj_df.drop(['speed_mean','speed_std','curvature_mean', 'curvature_max', 'distance','n_points'])
Y=traj_df['label']

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

model = sklearn.ensemble.RandomForestClassifier(random_state=42)
model.fit(X_train, Y_train)
accuracy = model.score(X_test, Y_test)
print(f"Test Accuracy: {accuracy:.2f}")

KeyError: "['speed_mean', 'speed_std', 'curvature_mean', 'curvature_max', 'distance', 'n_points'] not found in axis"